# Обучение ruBERT-large для классификации направления сигнала ЦБ в Google Colab

Этот notebook обучает вторую модель пайплайна `telegram_signal_searcher`: модель определения направления сигнала ЦБ в Telegram-постах.

Основная постановка задачи:

```text
Input:  [TOPIC=t_i] text
Output: direction = "+" или "-"
```

Модель не определяет тему поста. Эту задачу решает отдельная модель тем. Здесь считаем, что тема `t_i` уже найдена, и отвечаем на более узкий вопрос: усиливает ли пост сигнал Центрального банка по этой теме или ослабляет его.

Notebook подготовлен для запуска в **Google Colab** с готовыми файлами обучения и тестирования:

```text
data/annotated_data/direction_train.parquet
data/annotated_data/direction_test.parquet
```

Перед обучением включи GPU:

```text
Runtime / Среда выполнения -> Change runtime type -> GPU
```

Файлы можно положить в Google Drive с той же структурой папок, загрузить в `/content/data/annotated_data/` или выбрать оба parquet-файла через форму загрузки Colab.


## 1. Постановка задачи

В проекте уже есть первая модель, которая решает задачу многометочной классификации тем:

```text
text -> t1_relevant, t2_relevant, t3_relevant, t4_relevant, t5_relevant
```

Теперь обучается вторая модель: модель направления.

```text
[TOPIC=t_i] text -> "+" / "-"
```

Будущий пайплайн выглядит так:

```text
1. Модель тем:
   text -> t1=1, t3=1

2. Модель направления:
   [TOPIC=t1] text -> "+"
   [TOPIC=t3] text -> "-"

3. Итог:
   post_uid, topic, topic_probability, direction, direction_probability
```

Важно разделять эти две задачи. Первая модель отвечает, о каких темах говорит пост. Вторая модель отвечает, какое направление у поста внутри конкретной темы. Один и тот же текст может относиться сразу к нескольким темам, а направление для этих тем может быть разным. Поэтому одна строка исходной таблицы после преобразования может стать несколькими обучающими примерами: один пример на каждую релевантную тему.

## 2. Почему задача решается после классификации тем

Модель направления применяется только после того, как модель тем уже нашла релевантные темы. Это нужно по двум причинам.

Во-первых, направление без темы неоднозначно. Один пост может одновременно подтверждать один сигнал ЦБ и спорить с другим. Во-вторых, так проще встроить модель в финальный пайплайн: сначала получаем список тем, затем для каждой темы отдельно запускаем модель направления.

Одна строка обучающего датасета здесь означает пару `пост × тема`, а не просто пост. Именно поэтому ниже мы переводим исходную wide-таблицу в long-формат.

## 3. Почему используется формат `[TOPIC=t_i] text`

ruBERT получает один текстовый вход. Чтобы модель понимала, для какой именно темы нужно определить направление, мы явно добавляем тему в начало текста:

```text
[TOPIC=t1] текст поста
[TOPIC=t3] тот же текст поста
```

Это позволяет использовать одну бинарную модель для всех пяти тем. Модель учится интерпретировать один и тот же пост по-разному в зависимости от токена темы. Такой формат также хорошо подходит для будущего инференса: модель тем выдаёт релевантные темы, а модель направления получает по одному запросу на каждую найденную тему.

## 4. Импорты и базовые настройки

В Colab нужно установить недостающие библиотеки. Ячейка ниже ставит `transformers`, `datasets`, `accelerate`, `scikit-learn` и `pyarrow`.

Если после установки Colab попросит перезапустить runtime, перезапусти и выполни notebook заново сверху.


In [ ]:
# Colab setup: установка зависимостей
%pip install -q -U pandas pyarrow transformers datasets accelerate scikit-learn

from pathlib import Path
import json
import os
import random

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)


Ниже собраны основные параметры эксперимента. По умолчанию notebook настроен под Google Colab.

`MODEL_SIZE = "large"` — основной вариант для итогового качества. Если Colab выдаёт CUDA out of memory или нужно быстро проверить пайплайн, временно переключи на `base`.

`CLASS_WEIGHT_MODE = "balanced"` включает веса классов для компенсации дисбаланса между `+` и `-`.


In [ ]:
# === Colab / local paths ===
# Основной режим: train/test уже разделены на уровне файлов.
# Если файлы лежат в Google Drive, оставь USE_GOOGLE_DRIVE = True.
# Если хочешь загрузить parquet напрямую через форму Colab, поставь False.
USE_GOOGLE_DRIVE = True
USE_EXISTING_SPLIT_COLUMN = False
SMOKE_RUN = False

TRAIN_FILE_NAME = "direction_train.parquet"
TEST_FILE_NAME = "direction_test.parquet"

COLAB_DATA_DIR = Path("/content/data/annotated_data")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/data/annotated_data")
LOCAL_DATA_DIR = Path("../data/annotated_data")

COLAB_TRAIN_PATH = COLAB_DATA_DIR / TRAIN_FILE_NAME
COLAB_TEST_PATH = COLAB_DATA_DIR / TEST_FILE_NAME
DRIVE_TRAIN_PATH = DRIVE_DATA_DIR / TRAIN_FILE_NAME
DRIVE_TEST_PATH = DRIVE_DATA_DIR / TEST_FILE_NAME
LOCAL_TRAIN_PATH = LOCAL_DATA_DIR / TRAIN_FILE_NAME
LOCAL_TEST_PATH = LOCAL_DATA_DIR / TEST_FILE_NAME

if IN_COLAB:
    WORK_DIR = Path("/content/direction_training")
    TRAIN_PATH = DRIVE_TRAIN_PATH if USE_GOOGLE_DRIVE else COLAB_TRAIN_PATH
    TEST_PATH = DRIVE_TEST_PATH if USE_GOOGLE_DRIVE else COLAB_TEST_PATH
else:
    WORK_DIR = Path("../outputs/direction_training")
    TRAIN_PATH = LOCAL_TRAIN_PATH
    TEST_PATH = LOCAL_TEST_PATH

DATASET_DIR = WORK_DIR / "direction_dataset"
ERROR_DIR = WORK_DIR / "direction_errors"

TOPICS = ["t1", "t2", "t3", "t4", "t5"]

RELEVANT_COLS = {
    "t1": "t1_relevant",
    "t2": "t2_relevant",
    "t3": "t3_relevant",
    "t4": "t4_relevant",
    "t5": "t5_relevant",
}

DIRECTION_COLS = {
    "t1": "t1_direction",
    "t2": "t2_direction",
    "t3": "t3_direction",
    "t4": "t4_direction",
    "t5": "t5_direction",
}

META_COLS = [
    "post_uid",
    "orig_row",
    "channel_id",
    "channel_name",
    "channel_username",
    "message_id",
    "post_id",
    "post_date",
    "date",
    "source_file",
    "direction_reasoning",
]

REQUIRED_LONG_COLUMNS = ["text", "topic", "direction"]

LABEL2ID = {"-": 0, "+": 1}
ID2LABEL = {0: "-", 1: "+"}

MODEL_SIZE = "large"  # "base" или "large"
CLASS_WEIGHT_MODE = "balanced"  # "balanced", "sqrt", "none"
CLASS_WEIGHT_CAP = 10.0

MAX_LENGTH = 512
TEXT_COL = "model_text"

VALID_SIZE = 0.15
SEED = 42

EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

DATASET_DIR.mkdir(parents=True, exist_ok=True)
ERROR_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("WORK_DIR:", WORK_DIR)
print("Initial TRAIN_PATH:", TRAIN_PATH)
print("Initial TEST_PATH:", TEST_PATH)


## 5. Загрузка данных

На этом шаге notebook ищет два входных файла:

```text
direction_train.parquet
direction_test.parquet
```

Процесс загрузки устроен так:

1. **Google Drive:** положи файлы в `MyDrive/data/annotated_data/` или оставь их в проектной папке на Drive. Notebook сначала проверит явный путь, затем попробует найти файлы в `MyDrive`.
2. **Загрузка с компьютера:** поставь `USE_GOOGLE_DRIVE = False` в настройках выше и запусти ячейку ниже. Если файлы не найдены в `/content`, Colab предложит выбрать оба parquet-файла.
3. **Репозиторий в Colab:** если структура `data/annotated_data/` уже есть внутри `/content`, auto-search тоже её найдёт.

Результат этой ячейки — два DataFrame: `train_full_df` для обучения и валидации, а также `test_df` для финальной оценки модели.


In [ ]:
def mount_drive_if_needed():
    if IN_COLAB and USE_GOOGLE_DRIVE and not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")


def find_file(filename, roots, skip_dir_names=None):
    skip_dir_names = set(skip_dir_names or [])
    matches = []

    for root in roots:
        root_path = Path(root)
        if not root_path.exists():
            continue

        for current_dir, dirnames, filenames in os.walk(root_path):
            dirnames[:] = [name for name in dirnames if name not in skip_dir_names]
            if filename in filenames:
                matches.append(Path(current_dir) / filename)

    if not matches:
        searched = ", ".join(str(Path(root)) for root in roots)
        raise FileNotFoundError(f"Could not find {filename}. Searched: {searched}")

    matches = sorted(set(matches), key=lambda path: len(str(path)))

    if len(matches) > 1:
        print(f"Found several candidates for {filename}:")
        for match in matches:
            print(" -", match)

    return matches[0]


def upload_files_in_colab(expected_filenames):
    from google.colab import files

    print("Не удалось найти входные parquet-файлы автоматически.")
    print("Выбери оба файла:", ", ".join(expected_filenames))
    uploaded = files.upload()
    uploaded_names = set(uploaded.keys())

    resolved = {}
    for filename in expected_filenames:
        uploaded_path = Path("/content") / filename
        if filename in uploaded_names and uploaded_path.exists():
            resolved[filename] = uploaded_path

    missing = [filename for filename in expected_filenames if filename not in resolved]
    if missing:
        raise FileNotFoundError(
            f"Uploaded files: {sorted(uploaded_names)}. Missing expected files: {missing}."
        )

    return resolved


def resolve_train_test_paths():
    configured_paths = {
        TRAIN_FILE_NAME: Path(TRAIN_PATH),
        TEST_FILE_NAME: Path(TEST_PATH),
    }
    resolved = {}

    for filename, path in configured_paths.items():
        if path.exists():
            resolved[filename] = path

    missing = [filename for filename in configured_paths if filename not in resolved]

    if missing:
        local_search_roots = [COLAB_DATA_DIR, Path("/content"), LOCAL_DATA_DIR, Path(".")]
        for filename in list(missing):
            try:
                resolved[filename] = find_file(filename, local_search_roots, skip_dir_names={"drive"})
                missing.remove(filename)
            except FileNotFoundError:
                pass

    if missing and IN_COLAB and USE_GOOGLE_DRIVE:
        mount_drive_if_needed()
        drive_search_roots = [DRIVE_DATA_DIR, Path("/content/drive/MyDrive")]
        for filename in list(missing):
            try:
                resolved[filename] = find_file(filename, drive_search_roots)
                missing.remove(filename)
            except FileNotFoundError:
                pass

    if missing and IN_COLAB:
        uploaded_paths = upload_files_in_colab(missing)
        resolved.update(uploaded_paths)
        missing = [filename for filename in configured_paths if filename not in resolved]

    if missing:
        raise FileNotFoundError(f"Missing input files: {missing}")

    return resolved[TRAIN_FILE_NAME], resolved[TEST_FILE_NAME]


TRAIN_PATH, TEST_PATH = resolve_train_test_paths()

train_full_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("Resolved TRAIN_PATH:", TRAIN_PATH)
print("Resolved TEST_PATH:", TEST_PATH)
print("train_full_df shape:", train_full_df.shape)
print("test_df shape:", test_df.shape)
print("train columns:", train_full_df.columns.tolist())
print("test columns:", test_df.columns.tolist())
display(train_full_df.head())
display(test_df.head())
train_full_df.info()
test_df.info()


Ожидаемые варианты схемы:

**Long-формат:**

```text
text | topic | direction | label_id | model_text
```

**Wide-формат:**

```text
text
t1_relevant ... t5_relevant
t1_direction ... t5_direction
```

`direction_train.parquet` и `direction_test.parquet` уже задают разделение на обучение и тест, но сами таблицы могут быть в wide-формате. Notebook автоматически определяет формат и при необходимости собирает long-таблицу `post_uid | text | topic | direction | label_id | model_text`.

Если `post_uid` отсутствует, он создаётся из `channel_id + message_id`, из `orig_row` или из индекса строки. Это нужно для корректного валидационного разбиения без утечки данных.

Итог процесса: каждый релевантный пост превращается в одну или несколько строк — по одной строке на каждую тему, для которой известно направление `+` или `-`.


In [ ]:
def normalize_direction(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    plus_values = {
        "+",
        "plus",
        "positive",
        "pos",
        "up",
        "вверх",
        "плюс",
        "1",
    }

    minus_values = {
        "-",
        "minus",
        "negative",
        "neg",
        "down",
        "вниз",
        "минус",
        "0",
    }

    if value in plus_values:
        return "+"

    if value in minus_values:
        return "-"

    return np.nan


def is_relevant(value):
    if pd.isna(value):
        return False

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(value) == 1

    value = str(value).strip().lower()
    return value in {"1", "true", "yes", "y", "да", "+", "relevant"}


def build_post_uid(frame):
    frame = frame.copy()

    if "post_uid" not in frame.columns:
        if {"channel_id", "message_id"}.issubset(frame.columns):
            frame["post_uid"] = frame["channel_id"].astype(str) + "_" + frame["message_id"].astype(str)
        elif "orig_row" in frame.columns:
            frame["post_uid"] = frame["orig_row"].astype(str)
        else:
            frame["post_uid"] = np.arange(len(frame)).astype(str)

    frame["post_uid"] = frame["post_uid"].astype(str)
    return frame


def wide_to_long_direction(frame, name):
    frame = build_post_uid(frame)

    if "text" not in frame.columns:
        raise ValueError(f"{name}: missing required column 'text'")

    frame["text"] = frame["text"].fillna("").astype(str).str.strip()
    frame = frame[frame["text"].str.len() > 0].copy()

    available_meta_cols = [column for column in META_COLS if column in frame.columns]
    long_parts = []

    for topic in TOPICS:
        relevant_col = RELEVANT_COLS[topic]
        direction_col = DIRECTION_COLS[topic]

        if direction_col not in frame.columns:
            print(f"{name}: skip {topic}, missing {direction_col}")
            continue

        topic_df = frame.copy()
        topic_df["topic"] = topic
        topic_df["direction"] = topic_df[direction_col].apply(normalize_direction)
        topic_df = topic_df[topic_df["direction"].isin(["+", "-"])].copy()

        if relevant_col in topic_df.columns:
            relevant_mask = topic_df[relevant_col].apply(is_relevant)
            if relevant_mask.sum() > 0:
                topic_df = topic_df[relevant_mask].copy()
            else:
                print(
                    f"{name}: {relevant_col} produced 0 truthy rows; "
                    f"keeping rows with valid {direction_col} only"
                )

        if topic_df.empty:
            continue

        topic_df["label_id"] = topic_df["direction"].map(LABEL2ID).astype(int)
        topic_df[TEXT_COL] = "[TOPIC=" + topic_df["topic"].astype(str) + "] " + topic_df["text"].astype(str)

        keep_cols = available_meta_cols + ["text", TEXT_COL, "topic", "direction", "label_id"]
        keep_cols = list(dict.fromkeys(keep_cols))
        long_parts.append(topic_df[keep_cols])

    if not long_parts:
        raise ValueError(
            f"{name}: could not build long dataset. "
            f"Expected columns like t1_direction ... t5_direction."
        )

    long_frame = pd.concat(long_parts, ignore_index=True).reset_index(drop=True)

    print(f"{name}: detected WIDE format and converted to LONG")
    print(f"{name}: source rows={len(frame)}")
    print(f"{name}: long rows={len(long_frame)}")
    print(f"{name}: unique post_uid={long_frame['post_uid'].nunique()}")

    return long_frame


def prepare_long_schema(frame, name):
    frame = frame.copy()

    missing_columns = [column for column in REQUIRED_LONG_COLUMNS if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"{name}: missing required long columns: {missing_columns}")

    before = len(frame)

    frame = build_post_uid(frame)
    frame["text"] = frame["text"].fillna("").astype(str).str.strip()
    frame["topic"] = frame["topic"].astype(str).str.strip()
    frame["direction"] = frame["direction"].apply(normalize_direction)
    frame["label_id"] = frame["direction"].map(LABEL2ID)

    if TEXT_COL not in frame.columns:
        frame[TEXT_COL] = "[TOPIC=" + frame["topic"].astype(str) + "] " + frame["text"].astype(str)
    else:
        frame[TEXT_COL] = frame[TEXT_COL].fillna("").astype(str).str.strip()
        missing_model_text = frame[TEXT_COL].str.len() == 0
        frame.loc[missing_model_text, TEXT_COL] = (
            "[TOPIC="
            + frame.loc[missing_model_text, "topic"].astype(str)
            + "] "
            + frame.loc[missing_model_text, "text"].astype(str)
        )

    frame = frame[
        frame["topic"].isin(TOPICS)
        & frame["direction"].isin(["+", "-"])
        & frame["label_id"].isin([0, 1])
        & (frame["text"].str.len() > 0)
        & (frame[TEXT_COL].str.len() > 0)
    ].copy()

    frame["label_id"] = frame["label_id"].astype(int)
    frame["post_uid"] = frame["post_uid"].astype(str)
    frame = frame.reset_index(drop=True)

    after = len(frame)
    print(f"{name}: detected LONG format")
    print(f"{name}: rows before={before}, after={after}, dropped={before - after}")
    print(f"{name}: unique post_uid={frame['post_uid'].nunique()}")

    if frame.empty:
        raise ValueError(f"{name}: no rows left after schema normalization")

    return frame


def prepare_direction_schema(frame, name):
    print(f"{name}: columns:")
    print(frame.columns.tolist())

    has_long_columns = set(REQUIRED_LONG_COLUMNS).issubset(frame.columns)
    has_wide_direction_columns = any(
        direction_col in frame.columns
        for direction_col in DIRECTION_COLS.values()
    )

    if has_long_columns:
        return prepare_long_schema(frame, name)

    if has_wide_direction_columns:
        return prepare_long_schema(wide_to_long_direction(frame, name), name)

    raise ValueError(
        f"{name}: unknown direction dataset format. "
        f"Expected long columns {REQUIRED_LONG_COLUMNS} or wide columns t1_direction ... t5_direction. "
        f"Available columns: {frame.columns.tolist()}"
    )


train_full_df = prepare_direction_schema(train_full_df, "train_full_df")
test_df = prepare_direction_schema(test_df, "test_df")

available_meta_cols = [column for column in META_COLS if column in test_df.columns]
print("Available meta columns for test predictions:", available_meta_cols)


## 6. Проверка подготовленных обучающих и тестовых данных

После auto-detect и возможного преобразования `wide -> long` обе таблицы должны содержать обучающие колонки:

```text
post_uid | text | topic | direction | label_id | model_text
```

`direction_test.parquet` остаётся отдельной тестовой выборкой. Валидационная выборка будет создана только из `direction_train.parquet` по `post_uid`.

Эта проверка сохраняет нормализованные версии train/test в `direction_dataset`, чтобы можно было отдельно посмотреть, какие данные реально пошли в обучение.


In [ ]:
for split_name, split_df in {
    "train_full": train_full_df,
    "test": test_df,
}.items():
    required_columns = ["post_uid", "text", "topic", "direction", "label_id", TEXT_COL]
    missing_columns = [column for column in required_columns if column not in split_df.columns]
    if missing_columns:
        raise ValueError(f"{split_name}: missing prepared columns: {missing_columns}")

    parquet_path = DATASET_DIR / f"direction_{split_name}_normalized.parquet"
    csv_path = DATASET_DIR / f"direction_{split_name}_normalized.csv"
    split_df.to_parquet(parquet_path, index=False)
    split_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    print(f"{split_name} shape:", split_df.shape)
    print(f"{split_name} unique post_uid:", split_df["post_uid"].nunique())
    print("Saved:", parquet_path)
    print("Saved:", csv_path)
    display(split_df.head())


## 7. Проверка распределений до валидационного разбиения

На этом этапе смотрим баланс классов `+` и `-`, а также распределение направлений по темам. Это быстрая диагностика качества входной разметки: если какая-то тема почти пустая или один класс сильно доминирует, это будет видно до запуска обучения.


In [ ]:
def topic_distribution(split_df):
    return pd.crosstab(split_df["topic"], split_df["direction"]).reindex(
        index=TOPICS,
        columns=["-", "+"],
        fill_value=0,
    )


print("Train direction distribution:")
display(train_full_df["direction"].value_counts().reindex(["-", "+"], fill_value=0))

print("Test direction distribution:")
display(test_df["direction"].value_counts().reindex(["-", "+"], fill_value=0))

print("Train topic x direction:")
display(topic_distribution(train_full_df))

print("Test topic x direction:")
display(topic_distribution(test_df))


## 8. Валидационное разбиение без утечки данных

`direction_train.parquet` используется как полный пул обучающих данных. Из него создаётся валидационная выборка по `post_uid`, чтобы один и тот же пост не попадал одновременно в обучение и валидацию под разными темами.

`direction_test.parquet` не пересэмплируется и остаётся финальной тестовой выборкой.

После split notebook сохраняет три подготовленных файла: `direction_train_prepared.parquet`, `direction_valid_prepared.parquet` и `direction_test_prepared.parquet`.


In [ ]:
def split_by_post_uid(frame, valid_size=VALID_SIZE, seed=SEED):
    unique_post_uids = frame["post_uid"].drop_duplicates()
    if len(unique_post_uids) < 2:
        raise ValueError("Need at least 2 unique post_uid values for train/valid split")

    train_uids, valid_uids = train_test_split(
        unique_post_uids,
        test_size=valid_size,
        random_state=seed,
        shuffle=True,
    )

    train_uid_set = set(train_uids.astype(str))
    valid_uid_set = set(valid_uids.astype(str))

    assert train_uid_set.isdisjoint(valid_uid_set)

    train_part = frame[frame["post_uid"].isin(train_uid_set)].reset_index(drop=True)
    valid_part = frame[frame["post_uid"].isin(valid_uid_set)].reset_index(drop=True)
    return train_part, valid_part


if USE_EXISTING_SPLIT_COLUMN and "split" in train_full_df.columns:
    split_values = train_full_df["split"].astype(str).str.lower().str.strip()
    train_df = train_full_df[split_values == "train"].reset_index(drop=True)
    valid_df = train_full_df[split_values.isin(["valid", "validation", "val"])].reset_index(drop=True)
    if train_df.empty or valid_df.empty:
        raise ValueError("Existing split column did not produce non-empty train and valid splits")
else:
    train_df, valid_df = split_by_post_uid(train_full_df, valid_size=VALID_SIZE, seed=SEED)

train_test_overlap = set(train_df["post_uid"]).intersection(set(test_df["post_uid"]))
valid_test_overlap = set(valid_df["post_uid"]).intersection(set(test_df["post_uid"]))
if train_test_overlap or valid_test_overlap:
    print("Предупреждение: найдено пересечение post_uid с тестовой выборкой")
    print("train/test overlap:", len(train_test_overlap))
    print("valid/test overlap:", len(valid_test_overlap))

if SMOKE_RUN:
    EPOCHS = 1
    train_df = train_df.sample(min(len(train_df), 200), random_state=SEED).reset_index(drop=True)
    valid_df = valid_df.sample(min(len(valid_df), 100), random_state=SEED).reset_index(drop=True)
    test_df = test_df.sample(min(len(test_df), 100), random_state=SEED).reset_index(drop=True)

prepared_splits = {
    "train": train_df,
    "valid": valid_df,
    "test": test_df,
}

for split_name, split_df in prepared_splits.items():
    parquet_path = DATASET_DIR / f"direction_{split_name}_prepared.parquet"
    csv_path = DATASET_DIR / f"direction_{split_name}_prepared.csv"
    split_df.to_parquet(parquet_path, index=False)
    split_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print("Saved:", parquet_path)
    print("Saved:", csv_path)

print("train_df shape:", train_df.shape)
print("valid_df shape:", valid_df.shape)
print("test_df shape:", test_df.shape)
print("train post_uid:", train_df["post_uid"].nunique())
print("valid post_uid:", valid_df["post_uid"].nunique())
print("test post_uid:", test_df["post_uid"].nunique())


def label_distribution_for_split(split_df, split_name):
    result = (
        split_df["direction"]
        .value_counts()
        .reindex(["-", "+"], fill_value=0)
        .rename_axis("direction")
        .reset_index(name="count")
    )
    result.insert(0, "split", split_name)
    return result


label_distribution_by_split = pd.concat(
    [
        label_distribution_for_split(train_df, "train"),
        label_distribution_for_split(valid_df, "valid"),
        label_distribution_for_split(test_df, "test"),
    ],
    ignore_index=True,
)

train_topic_distribution = topic_distribution(train_df)
valid_topic_distribution = topic_distribution(valid_df)
test_topic_distribution = topic_distribution(test_df)

display(label_distribution_by_split)
display(train_topic_distribution)
display(valid_topic_distribution)
display(test_topic_distribution)

label_distribution_by_split.to_csv(
    WORK_DIR / "direction_label_distribution_by_split.csv",
    index=False,
    encoding="utf-8-sig",
)
train_topic_distribution.to_csv(WORK_DIR / "direction_topic_distribution_train.csv", encoding="utf-8-sig")
valid_topic_distribution.to_csv(WORK_DIR / "direction_topic_distribution_valid.csv", encoding="utf-8-sig")
test_topic_distribution.to_csv(WORK_DIR / "direction_topic_distribution_test.csv", encoding="utf-8-sig")

print("Saved split distributions to:", WORK_DIR)


## 9. Настройка ruBERT base/large

`base` — быстрый базовый вариант, который удобен для проверки пайплайна и первичного сравнения.

`large` — основной более тяжёлый эксперимент для итогового качества. Для него нужен GPU, `fp16`, маленький batch size и gradient checkpointing. TPU здесь не используется: notebook рассчитан на обычный PyTorch GPU в Google Colab.


In [ ]:
if MODEL_SIZE == "base":
    MODEL_NAME = "ai-forever/ruBert-base"
elif MODEL_SIZE == "large":
    MODEL_NAME = "ai-forever/ruBert-large"
else:
    raise ValueError("MODEL_SIZE must be 'base' or 'large'")

if MODEL_SIZE == "large":
    PER_DEVICE_TRAIN_BATCH_SIZE = 1
    PER_DEVICE_EVAL_BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 16
    FP16 = True
    GRADIENT_CHECKPOINTING = True
else:
    PER_DEVICE_TRAIN_BATCH_SIZE = 8
    PER_DEVICE_EVAL_BATCH_SIZE = 16
    GRADIENT_ACCUMULATION_STEPS = 2
    FP16 = torch.cuda.is_available()
    GRADIENT_CHECKPOINTING = False

print("MODEL_NAME:", MODEL_NAME)
print("PER_DEVICE_TRAIN_BATCH_SIZE:", PER_DEVICE_TRAIN_BATCH_SIZE)
print("PER_DEVICE_EVAL_BATCH_SIZE:", PER_DEVICE_EVAL_BATCH_SIZE)
print("GRADIENT_ACCUMULATION_STEPS:", GRADIENT_ACCUMULATION_STEPS)
print("FP16:", FP16)
print("GRADIENT_CHECKPOINTING:", GRADIENT_CHECKPOINTING)

## 10. Диагностика CUDA

Эта ячейка проверяет, видит ли PyTorch GPU. Она не устанавливает CUDA-пакеты и не меняет окружение. Если `cuda available` равно `False`, ruBERT-large лучше не запускать.

In [ ]:
print("torch:", torch.__version__)
print("cuda version:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

if torch.cuda.is_available():
    x = torch.tensor([1.0, 2.0, 3.0], device="cuda")
    y = x * 2
    print("CUDA smoke test:", y)

## 11. Токенизация и динамический padding

Мы используем динамический padding, а не фиксированное дополнение всех текстов до 512 токенов. Это особенно важно для ruBERT-large: каждый batch дополняется только до максимальной длины текста внутри этого batch, что снижает расход GPU-памяти и ускоряет обучение.

Максимальная длина — `512`, потому что это стандартный предел для BERT-подобных моделей. Более длинные Telegram-посты будут обрезаны.

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
)

## 12. Датасеты Hugging Face

В отличие от многометочной модели тем, здесь задача бинарная и однозначная: для одной пары `пост × тема` есть только один правильный класс. Поэтому используется `softmax` и `CrossEntropyLoss`, а метка хранится как `int64`: `0` для `-`, `1` для `+`.

После токенизации исходный текст `model_text` удаляется из датасета Hugging Face, потому что для обучения нужны только `input_ids`, `attention_mask` и `labels`. Полные тексты остаются в pandas DataFrame для анализа ошибок и сохранения предсказаний.

In [ ]:
from datasets import Dataset


def make_hf_dataset(split_df):
    hf_df = split_df[[TEXT_COL, "label_id"]].rename(columns={"label_id": "labels"}).copy()
    hf_df["labels"] = hf_df["labels"].astype(int)
    return Dataset.from_pandas(hf_df, preserve_index=False)


train_ds = make_hf_dataset(train_df)
valid_ds = make_hf_dataset(valid_df)
test_ds = make_hf_dataset(test_df)

train_ds = train_ds.map(tokenize_batch, batched=True)
valid_ds = valid_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

train_ds = train_ds.remove_columns([TEXT_COL])
valid_ds = valid_ds.remove_columns([TEXT_COL])
test_ds = test_ds.remove_columns([TEXT_COL])

train_ds.set_format("torch")
valid_ds.set_format("torch")
test_ds.set_format("torch")

sample = train_ds[0]
print(sample.keys())
print(sample["labels"])
print(sample["labels"].dtype)

assert sample["labels"].dtype == torch.int64

## 13. Загрузка модели

Для задачи определения направления используется `AutoModelForSequenceClassification` с `num_labels=2`. Это обычная бинарная классификация с двумя взаимоисключающими классами, поэтому здесь используется `softmax`, а не `sigmoid`.

Это отличается от первой модели тем: там у одного поста может быть несколько тем одновременно, поэтому нужна многометочная постановка. Здесь для конкретной пары `пост × тема` направление ровно одно: `+` или `-`.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

print(model.config.id2label)

## 14. Веса классов

Веса классов компенсируют дисбаланс между `+` и `-`. По умолчанию используется `balanced`-взвешивание, но для устойчивости предусмотрены режимы `sqrt` и `none`.

Веса передаются в `CrossEntropyLoss`: это помогает модели не игнорировать более редкий класс, если один тип направления встречается заметно реже.

In [ ]:
class_counts = train_df["label_id"].value_counts().sort_index()
n_classes = 2
n_samples = len(train_df)

raw_weights = n_samples / (n_classes * class_counts)
raw_weights = raw_weights.reindex([0, 1]).fillna(1.0)

if CLASS_WEIGHT_MODE == "balanced":
    used_weights = raw_weights
elif CLASS_WEIGHT_MODE == "sqrt":
    used_weights = np.sqrt(raw_weights)
elif CLASS_WEIGHT_MODE == "none":
    used_weights = pd.Series([1.0, 1.0], index=[0, 1])
else:
    raise ValueError("Unknown CLASS_WEIGHT_MODE")

used_weights = used_weights.clip(upper=CLASS_WEIGHT_CAP)

class_weights = torch.tensor(used_weights.values, dtype=torch.float32)

class_weights_df = pd.DataFrame(
    {
        "label_id": [0, 1],
        "label": [ID2LABEL[0], ID2LABEL[1]],
        "class_count": class_counts.reindex([0, 1]).fillna(0).astype(int).values,
        "raw_weight": raw_weights.values,
        "used_weight": used_weights.values,
    }
)

display(class_weights_df)
class_weights_df.to_csv(WORK_DIR / "direction_class_weights.csv", index=False, encoding="utf-8-sig")

print("class_weights tensor:", class_weights)
print("Saved:", WORK_DIR / "direction_class_weights.csv")

## 15. Trainer с весами для бинарной классификации

Так как это бинарная классификация с одним правильным классом, используется `CrossEntropyLoss`. В неё можно передать веса классов, чтобы модель не игнорировала менее частый класс.

Мы не используем `BCEWithLogitsLoss`, потому что классы взаимоисключающие: для одной пары `пост × тема` правильный ответ либо `-`, либо `+`, но не оба одновременно.

In [ ]:
from transformers import Trainer


class WeightedDirectionTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").long()
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )

        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

## 16. Метрики

Главная метрика — `macro F1`, потому что важно качество по обоим направлениям, а не только по более частому классу. Accuracy может быть обманчивой при дисбалансе: модель может хорошо угадывать частый класс и плохо работать с редким.

Дополнительно считаются ROC-AUC и PR-AUC по вероятности класса `+`. PR-AUC особенно полезна, если положительный класс встречается реже.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    f1_minus = f1_score(labels, preds, pos_label=0, zero_division=0)
    f1_plus = f1_score(labels, preds, pos_label=1, zero_division=0)

    metrics = {
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": precision_macro,
        "macro_recall": recall_macro,
        "macro_f1": f1_macro,
        "weighted_precision": precision_weighted,
        "weighted_recall": recall_weighted,
        "weighted_f1": f1_weighted,
        "f1_minus": f1_minus,
        "f1_plus": f1_plus,
    }

    try:
        metrics["roc_auc"] = roc_auc_score(labels, probs[:, 1])
    except ValueError:
        metrics["roc_auc"] = 0.0

    try:
        metrics["pr_auc_plus"] = average_precision_score(labels, probs[:, 1])
    except ValueError:
        metrics["pr_auc_plus"] = 0.0

    return metrics

## 17. Параметры обучения для base / large

Для `large` используется маленький batch size и большой `gradient_accumulation_steps`, чтобы поместить модель в память Colab GPU. Gradient checkpointing снижает расход GPU-памяти, что особенно важно для ruBERT-large. Цена — небольшое замедление обучения, потому что часть активаций пересчитывается во время backward pass.

В разных версиях `transformers` параметр evaluation strategy назывался по-разному. Helper ниже сначала пробует `eval_strategy`, а если версия старая — автоматически переключается на `evaluation_strategy`.


In [ ]:
from transformers import TrainingArguments


def make_training_args(**kwargs):
    try:
        return TrainingArguments(**kwargs)
    except TypeError as error:
        if "eval_strategy" in kwargs:
            kwargs = dict(kwargs)
            kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
            return TrainingArguments(**kwargs)
        raise error


training_args = make_training_args(
    output_dir=str(WORK_DIR / f"rubert_{MODEL_SIZE}_direction_binary"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=20,
    save_total_limit=2,
    report_to="none",
    fp16=FP16,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    optim="adamw_torch",
    seed=SEED,
    data_seed=SEED,
)

training_args

## 18. Обучение модели

Следующая ячейка создаёт `Trainer`, а ячейка после неё запускает обучение. Выполнять обучение стоит в Colab с включённым GPU. Для ruBERT-large рекомендуется GPU T4/P100/A100, `fp16` и gradient checkpointing.


In [ ]:
trainer = WeightedDirectionTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

Эта ячейка запускает обучение. Выполнять её стоит только после включения GPU в Colab.

Проверка: в предыдущей CUDA-ячейке должно быть `cuda available: True`.


In [ ]:
train_result = trainer.train()

## 19. Оценка качества на тестовой выборке

После обучения оцениваем лучшую модель на тестовой выборке. Сохраняем не только агрегированные метрики, но и построчные предсказания с вероятностями `prob_minus` и `prob_plus`. Это нужно для дальнейшего анализа ошибок и для понимания уверенности модели.

В выходной CSV попадают meta columns, если они были в исходном датасете, а также `text`, `model_text`, `topic`, истинная разметка и предсказание.

In [ ]:
test_output = trainer.predict(test_ds)

test_logits = test_output.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=1).numpy()

test_pred_ids = np.argmax(test_probs, axis=1)
test_true_ids = test_df["label_id"].values

test_df_eval = test_df.copy()
test_df_eval["prob_minus"] = test_probs[:, 0]
test_df_eval["prob_plus"] = test_probs[:, 1]
test_df_eval["pred_label_id"] = test_pred_ids
test_df_eval["pred_direction"] = test_df_eval["pred_label_id"].map(ID2LABEL)
test_df_eval["is_correct"] = test_df_eval["label_id"] == test_df_eval["pred_label_id"]
test_df_eval["confidence"] = np.max(test_probs, axis=1)

precision_macro, recall_macro, _, _ = precision_recall_fscore_support(
    test_true_ids, test_pred_ids, average="macro", zero_division=0
)
precision_weighted, recall_weighted, _, _ = precision_recall_fscore_support(
    test_true_ids, test_pred_ids, average="weighted", zero_division=0
)

test_metrics = {
    "accuracy": float(accuracy_score(test_true_ids, test_pred_ids)),
    "macro_precision": float(precision_macro),
    "macro_recall": float(recall_macro),
    "macro_f1": float(f1_score(test_true_ids, test_pred_ids, average="macro", zero_division=0)),
    "weighted_precision": float(precision_weighted),
    "weighted_recall": float(recall_weighted),
    "weighted_f1": float(f1_score(test_true_ids, test_pred_ids, average="weighted", zero_division=0)),
    "f1_minus": float(f1_score(test_true_ids, test_pred_ids, pos_label=0, zero_division=0)),
    "f1_plus": float(f1_score(test_true_ids, test_pred_ids, pos_label=1, zero_division=0)),
}

try:
    test_metrics["roc_auc"] = float(roc_auc_score(test_true_ids, test_probs[:, 1]))
except ValueError:
    test_metrics["roc_auc"] = 0.0

try:
    test_metrics["pr_auc_plus"] = float(average_precision_score(test_true_ids, test_probs[:, 1]))
except ValueError:
    test_metrics["pr_auc_plus"] = 0.0

report_dict = classification_report(
    test_true_ids,
    test_pred_ids,
    target_names=["-", "+"],
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).T

cm = confusion_matrix(test_true_ids, test_pred_ids, labels=[0, 1])
cm_df = pd.DataFrame(cm, index=["true_-", "true_+"], columns=["pred_-", "pred_+"])

prediction_cols = [
    column
    for column in available_meta_cols
    + [
        "text",
        "model_text",
        "topic",
        "direction",
        "label_id",
        "prob_minus",
        "prob_plus",
        "pred_label_id",
        "pred_direction",
        "is_correct",
        "confidence",
    ]
    if column in test_df_eval.columns
]
prediction_cols = list(dict.fromkeys(prediction_cols))

test_df_eval[prediction_cols].to_csv(WORK_DIR / "direction_test_predictions.csv", index=False, encoding="utf-8-sig")
report_df.to_csv(WORK_DIR / "direction_classification_report.csv", encoding="utf-8-sig")
cm_df.to_csv(WORK_DIR / "direction_confusion_matrix.csv", encoding="utf-8-sig")

with open(WORK_DIR / "direction_metrics.json", "w", encoding="utf-8") as file:
    json.dump(test_metrics, file, ensure_ascii=False, indent=2)

display(pd.DataFrame([test_metrics]))
display(report_df)
display(cm_df)

print("Saved test artifacts to:", WORK_DIR)

## 20. Метрики по темам

Общая метрика может скрывать проблемы на отдельных темах. Например, модель может хорошо работать на `t1`, но путать направление на `t4`, если там мало примеров или сложная риторика Telegram-каналов.

Поэтому отдельно считаются метрики для каждой темы `t1`–`t5`.

In [ ]:
def safe_binary_count(values, label_id):
    return int((np.asarray(values) == label_id).sum())


topic_metric_rows = []

for topic in TOPICS:
    topic_part = test_df_eval[test_df_eval["topic"] == topic].copy()

    if topic_part.empty:
        topic_metric_rows.append(
            {
                "topic": topic,
                "n": 0,
                "accuracy": 0.0,
                "macro_f1": 0.0,
                "weighted_f1": 0.0,
                "f1_minus": 0.0,
                "f1_plus": 0.0,
                "true_plus": 0,
                "true_minus": 0,
                "pred_plus": 0,
                "pred_minus": 0,
            }
        )
        continue

    y_true = topic_part["label_id"].values
    y_pred = topic_part["pred_label_id"].values

    topic_metric_rows.append(
        {
            "topic": topic,
            "n": int(len(topic_part)),
            "accuracy": float(accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
            "f1_minus": float(f1_score(y_true, y_pred, pos_label=0, zero_division=0)),
            "f1_plus": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
            "true_plus": safe_binary_count(y_true, 1),
            "true_minus": safe_binary_count(y_true, 0),
            "pred_plus": safe_binary_count(y_pred, 1),
            "pred_minus": safe_binary_count(y_pred, 0),
        }
    )

topic_metrics_df = pd.DataFrame(topic_metric_rows)
topic_metrics_df.to_csv(WORK_DIR / "direction_topic_metrics.csv", index=False, encoding="utf-8-sig")

display(topic_metrics_df)
print("Saved:", WORK_DIR / "direction_topic_metrics.csv")

## 21. Анализ ошибок

Анализ ошибок нужен для ручной проверки, какие именно посты модель путает: усиливающие сигнал как ослабляющие или наоборот. Отдельные файлы по темам позволяют понять, какие сигналы ЦБ сложнее всего интерпретируются Telegram-каналами.

Ошибки сортируются по теме, типу ошибки и уверенности модели. Самые уверенные ошибки особенно полезны: они часто показывают системные слабости разметки, модели или самой постановки задачи.

In [ ]:
errors_df = test_df_eval[test_df_eval["is_correct"] == False].copy()

errors_df["error_type"] = np.where(
    (errors_df["direction"] == "+") & (errors_df["pred_direction"] == "-"),
    "plus_predicted_as_minus",
    "minus_predicted_as_plus",
)

if not errors_df.empty:
    errors_df = errors_df.sort_values(["topic", "error_type", "confidence"], ascending=[True, True, False])

errors_df.to_csv(ERROR_DIR / "direction_errors_all.csv", index=False, encoding="utf-8-sig")

if errors_df.empty:
    error_summary = pd.DataFrame(columns=["topic", "error_type", "n", "mean_confidence", "max_confidence"])
else:
    error_summary = (
        errors_df.groupby(["topic", "error_type"])
        .agg(
            n=("post_uid", "size"),
            mean_confidence=("confidence", "mean"),
            max_confidence=("confidence", "max"),
        )
        .reset_index()
        .sort_values(["topic", "error_type"])
    )

error_summary.to_csv(ERROR_DIR / "direction_error_summary.csv", index=False, encoding="utf-8-sig")

errors_df[errors_df["error_type"] == "plus_predicted_as_minus"].to_csv(
    ERROR_DIR / "plus_predicted_as_minus.csv", index=False, encoding="utf-8-sig"
)
errors_df[errors_df["error_type"] == "minus_predicted_as_plus"].to_csv(
    ERROR_DIR / "minus_predicted_as_plus.csv", index=False, encoding="utf-8-sig"
)

for topic in TOPICS:
    topic_errors = errors_df[errors_df["topic"] == topic].sort_values("confidence", ascending=False)
    topic_errors.to_csv(ERROR_DIR / f"{topic}_direction_errors.csv", index=False, encoding="utf-8-sig")

display(errors_df.head(30))
display(error_summary)

for topic in TOPICS:
    print(f"Top confident errors for {topic}")
    display(errors_df[errors_df["topic"] == topic].sort_values("confidence", ascending=False).head(5))

print("Saved error analysis files to:", ERROR_DIR)

## 22. Сохранение модели

Финальная модель сохраняется в папку:

```text
/content/direction_training/rubert_<base|large>_direction_binary_<class_weight_mode>_final
```

Эта папка потом попадёт в архив `direction_output.zip`.


In [ ]:
FINAL_MODEL_DIR = WORK_DIR / f"rubert_{MODEL_SIZE}_direction_binary_{CLASS_WEIGHT_MODE}_final"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

direction_model_config = {
    "model_name": MODEL_NAME,
    "model_size": MODEL_SIZE,
    "task": "binary_topic_direction_classification",
    "input_format": "[TOPIC=t_i] text",
    "data_input_mode": "presplit_train_test",
    "train_path": str(TRAIN_PATH),
    "test_path": str(TEST_PATH),
    "text_col": TEXT_COL,
    "source_text_col": "text",
    "topic_col": "topic",
    "label_col": "direction",
    "label2id": LABEL2ID,
    "id2label": ID2LABEL,
    "max_length": MAX_LENGTH,
    "topics": TOPICS,
    "class_weight_mode": CLASS_WEIGHT_MODE,
    "class_weight_cap": CLASS_WEIGHT_CAP,
    "metric_for_best_model": "macro_f1",
    "valid_size_from_train": VALID_SIZE,
    "use_existing_split_column": USE_EXISTING_SPLIT_COLUMN,
    "random_state": SEED,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
}

with open(FINAL_MODEL_DIR / "direction_model_config.json", "w", encoding="utf-8") as file:
    json.dump(direction_model_config, file, ensure_ascii=False, indent=2)

print("Final model directory:", FINAL_MODEL_DIR)
print("Files:")
for path in sorted(FINAL_MODEL_DIR.iterdir()):
    print(" -", path.name)


## 23. Как применять модель после модели тем

Эта функция нужна уже после обучения: она превращает выход topic-модели в формат, который понимает direction-модель.

Ожидаемый вход: DataFrame с постами и предсказанными темами, например:

```text
post_uid
text
t1_relevant_pred
t2_relevant_pred
...
t5_relevant_pred
```

Для всех тем, где `t_i_relevant_pred == 1`, создаётся строка:

```text
post_uid | text | topic | model_text
```

In [ ]:
def build_direction_inference_df(posts_df, text_col="text", topic_pred_suffix="_pred"):
    rows = []

    for _, row in posts_df.iterrows():
        for topic in TOPICS:
            pred_col = f"{topic}_relevant{topic_pred_suffix}"

            if pred_col not in posts_df.columns:
                continue

            value = row[pred_col]
            if pd.isna(value):
                continue

            try:
                is_topic_predicted = int(value) == 1
            except (TypeError, ValueError):
                is_topic_predicted = str(value).strip().lower() in {"1", "true", "yes", "y", "да"}

            if is_topic_predicted:
                text = str(row[text_col]).strip()
                item = {
                    "post_uid": row["post_uid"] if "post_uid" in posts_df.columns else None,
                    "text": text,
                    "topic": topic,
                    "model_text": f"[TOPIC={topic}] {text}",
                }
                rows.append(item)

    return pd.DataFrame(rows)

## 24. Ожидаемые выходные файлы после Colab-запуска

После полного запуска notebook сохраняет результаты в:

```text
/content/direction_training
```

Главные артефакты процесса:

```text
direction_dataset/direction_train_full_normalized.parquet
direction_dataset/direction_test_normalized.parquet
direction_dataset/direction_train_prepared.parquet
direction_dataset/direction_valid_prepared.parquet
direction_dataset/direction_test_prepared.parquet
direction_label_distribution_by_split.csv
direction_topic_distribution_train.csv
direction_topic_distribution_valid.csv
direction_topic_distribution_test.csv
direction_class_weights.csv
direction_test_predictions.csv
direction_classification_report.csv
direction_confusion_matrix.csv
direction_metrics.json
direction_topic_metrics.csv
direction_errors/
rubert_<base|large>_direction_binary_<class_weight_mode>_final/
```

Последняя ячейка упаковывает все результаты в архив `direction_output.zip`, который можно скачать из файловой панели Colab.


In [ ]:
!cd /content && zip -r direction_output.zip direction_training
print("Archive saved to: /content/direction_output.zip")
